In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# %sql

# -- ============================================
# -- Build each section as a single-row string, then combine
# -- ============================================

# WITH core AS (
#   SELECT 
#     hammer_price, number_of_lots, lots_sold, registered_bidders, winning_bidders,auction
#   FROM company_db.gold.auction_sales_report_view
#   WHERE start_date = :id
# ),

# units AS (
#   SELECT 
#     SUM(quantity) AS number_of_units,
#     ROUND(SUM(hammerprice) / NULLIF(SUM(quantity), 0), 0) AS avg_price_per_unit
#   FROM company_db.gold.auction_results_fact
#   WHERE start_date = :id
# ),

# new_bidders_count AS (
#   SELECT COUNT(DISTINCT user_id) AS new_bidders
#   FROM company_db.gold.bid_fact b
#   WHERE start_date = :id AND user_id IS NOT NULL
#     AND NOT EXISTS (
#       SELECT 1 FROM company_db.gold.bid_fact c
#       WHERE b.user_id = c.user_id AND c.start_date < :id
#     )
# ),

# no_show AS (
#   SELECT registered_bidders - winning_bidders AS no_show
#   FROM company_db.gold.auction_sales_report_view
#   WHERE batch_id = :id
# ),

# estimates_binned AS (
#   SELECT 
#     CASE
#       WHEN hammerprice > highestimate THEN 'ABOVE HIGH'
#       WHEN hammerprice < lowestimate THEN 'BELOW LOW'
#       ELSE 'IN RANGE'
#     END AS estimate_bucket
#   FROM company_db.gold.auction_results_fact
#   WHERE start_date = :id AND sold_flag = 1
# ),

# estimates AS (
#   SELECT
#     ROUND(100.0 * SUM(CASE WHEN estimate_bucket = 'ABOVE HIGH' THEN 1 ELSE 0 END) / COUNT(*), 0) AS pct_above_high,
#     ROUND(100.0 * SUM(CASE WHEN estimate_bucket = 'BELOW LOW' THEN 1 ELSE 0 END) / COUNT(*), 0) AS pct_below_low,
#     ROUND(100.0 * SUM(CASE WHEN estimate_bucket = 'IN RANGE' THEN 1 ELSE 0 END) / COUNT(*), 0) AS pct_in_range
#   FROM estimates_binned
# ),

# top_lots_rows AS (
#   SELECT 
#     CONCAT(
#       ROW_NUMBER() OVER (ORDER BY af.hammerprice DESC), '. ', ld.item_name,
#       ' — $', FORMAT_NUMBER(af.hammerprice, 0),
#       '  (est. $', FORMAT_NUMBER(af.lowestimate, 0), '–$', FORMAT_NUMBER(af.highestimate, 0), ')'
#     ) AS line
#   FROM company_db.gold.auction_results_fact af
#   INNER JOIN company_db.gold.lots_dim ld ON af.lot_dim_key = ld.lot_dim_key
#   WHERE af.start_date = :id
#   ORDER BY af.hammerprice DESC
#   LIMIT 3
# ),
# top_lots_text AS (
#   SELECT CONCAT_WS('\n', COLLECT_LIST(line)) AS lots_text FROM top_lots_rows
# ),

# top_spenders_rows AS (
#   SELECT 
#     CONCAT(
#       ROW_NUMBER() OVER (ORDER BY s.total_spend DESC), '. ', ud.full_name,
#       ' — $', FORMAT_NUMBER(s.total_spend, 0), ' (', s.number_of_lots, ' lots)'
#     ) AS line
#   FROM (
#     SELECT user_dim_key, SUM(hammerprice) AS total_spend, COUNT(lot_id) AS number_of_lots
#     FROM company_db.gold.auction_results_fact
#     WHERE start_date = :id
#     GROUP BY user_dim_key
#     ORDER BY total_spend DESC
#     LIMIT 3
#   ) s
#   INNER JOIN company_db.gold.users_dim ud ON s.user_dim_key = ud.user_dim_key
# ),
# top_spenders_text AS (
#   SELECT CONCAT_WS('\n', COLLECT_LIST(line)) AS spenders_text FROM top_spenders_rows
# ),

# new_bidder_spenders_agg AS (
#   SELECT 
#     ud.full_name,
#     SUM(arf.hammerprice) AS total_spend,
#     COUNT(arf.lot_id) AS number_of_lots
#   FROM (
#     SELECT DISTINCT user_id FROM company_db.gold.bid_fact b
#     WHERE start_date = :id AND user_id IS NOT NULL
#       AND NOT EXISTS (
#         SELECT 1 FROM company_db.gold.bid_fact c
#         WHERE b.user_id = c.user_id AND c.start_date < :id
#       )
#   ) nb
#   INNER JOIN company_db.gold.auction_results_fact arf ON nb.user_id = arf.user_id
#   INNER JOIN company_db.gold.users_dim ud ON nb.user_id = ud.user_id
#   WHERE arf.start_date = :id
#   GROUP BY ud.full_name
# ),
# new_bidder_spenders_rows AS (
#   SELECT 
#     CONCAT(
#       ROW_NUMBER() OVER (ORDER BY total_spend DESC), '. ', full_name,
#       ' — $', FORMAT_NUMBER(total_spend, 0), ' (', number_of_lots, ' lots)'
#     ) AS line
#   FROM new_bidder_spenders_agg
#   ORDER BY total_spend DESC
#   LIMIT 3
# ),
# new_spenders_text AS (
#   SELECT CONCAT_WS('\n', COLLECT_LIST(line)) AS new_spenders_text FROM new_bidder_spenders_rows
# )

# -- ============================================
# -- Final: assemble message and send
# -- ============================================
# SELECT http_request(
#   conn => 'slack_auction_conn',
#   method => 'POST',
#   path => 'services/<YOUR_SLACK_WEBHOOK_PATH>',
#   headers => map('Content-Type', 'application/json'),
#   json => to_json(named_struct(
#     'text', concat(
#       '🔨 *Auction Results: ', auction, '*\n\n',
#       '*SALES*\n',
#       'Hammered: $', FORMAT_NUMBER(core.hammer_price, 0), '\n',
#       'Lots: ', core.lots_sold, ' sold / ', core.number_of_lots, ' total   ',
# FORMAT_NUMBER(ROUND(100.0 * core.lots_sold / NULLIF(core.number_of_lots, 0), 1), 1), '% sale rate\n',

#       'Units: ', FORMAT_NUMBER(units.number_of_units, 0),
#       ' | Avg Value/Unit: $', FORMAT_NUMBER(units.avg_price_per_unit, 0), '\n\n',
#       '*BIDDERS*\n',
#       'Registered: ', core.registered_bidders, ' → Won: ', core.winning_bidders, '\n',
#       'New bidders: ', new_bidders_count.new_bidders, '\n',
#       'No-shows: ', no_show.no_show, '\n\n',
#       '*VS. ESTIMATES*\n',
#       'Above high: ', estimates.pct_above_high, '% | Below low: ', estimates.pct_below_low,
#       '% | In range: ', estimates.pct_in_range, '%\n\n',
#       '*TOP 3 SOLD LOTS*\n', top_lots_text.lots_text, '\n\n',
#       '*TOP SPENDERS*\n', top_spenders_text.spenders_text, '\n\n',
#       '*NEW BIDDERS — TOP SPENDERS*\n', new_spenders_text.new_spenders_text
#     )
#   ))
# )
# FROM core, units, new_bidders_count, no_show, estimates, top_lots_text, top_spenders_text, new_spenders_text;

In [ ]:
# %sql
# SELECT http_request(
#   conn => 'slack_auction_conn',
#   method => 'POST',
#   path => 'services/<YOUR_SLACK_WEBHOOK_PATH>',
#   headers => map('Content-Type', 'application/json'),
#   json => to_json(named_struct(
#     'text', concat(
#       '🔨 *Auction Results: ', auction, '*\n\n',
#       '📊 *Sales*\n',
#       '• Lots sold: ', lots_sold, ' / ', number_of_lots, 
#         ' (*', ROUND(lots_sold * 100.0 / number_of_lots, 1), '% sell-through)*\n',
#       '• Lots with bids: ', lots_with_bid, ' / ', number_of_lots, '\n',
#       '• Sold: ', lots_sold_online, ' lots\n',
#       '• Hammer price total: *$', format_number(hammer_price, 2), '*\n\n',
#       '👥 *Bidders*\n',
#       '• Registered: ', registered_bidders, 
#         ' → Approved: ', approved_bidders, 
#         ' → Won: ', winning_bidders, '\n',
#       '• Under-bidders: ', under_bidders, '\n',
#       '• Online bidders: ', online_bidders
#     )
#   ))
# )
# FROM company_db.gold.auction_sales_report_view
# WHERE batch_id = :id;

In [ ]:
# %sql
# SELECT * FROM company_db.gold.auction_sales_report_view

In [ ]:
# %sql
# SELECT 
# concat('💰 SALES',' ' ,format_number(hammer_price,2 )) as hammered,
# concat(lots_sold,' sold', ' / ',number_of_lots, ' total' ) as lots,
#         (
#         select * from
#                         ( select 
#                         concat('Units: ', sum(quantity))
#                         From 
#                         company_db.gold.auction_sales_report_view
#                         inner join company_db.gold.auction_results_fact
#                         on company_db.gold.auction_sales_report_view
#                         .auction_id = company_db.gold.auction_results_fact.auction_id
#                         where company_db.gold.auction_sales_report_view.auction_id = 12345) ) as number_of_units

# FROM 
# company_db.gold.auction_sales_report_view
# where auction_id = 12345


🔨 Auction Results: Fall Collectors Auction

💰 SALES
Hammered: $120,000  (+10% vs. last month, +5% vs. last year)  [illustrative]
Lots: 90 sold / 100 total  (90% sale rate)
Units: 1,000  |  Avg Value/Unit: $120

👥 BIDDERS
Registered: 200  →  Placed a bid: 180  →  Won: 60
  (+5% registered vs. last month)
🆕 New bidders (first auction): 25
⚠️ No-shows (registered, didn't bid): 20

📊 VS. ESTIMATES
Above high estimate: 20% of lots  |  Below low estimate: 10%
In range: 70%

🏆 TOP 3 SOLD LOTS
1. Lot A — $15,000  (est. $12,000–$18,000)
2. Lot B — $11,000  (est. $9,000–$13,000)
3. Lot C — $8,000  (est. $7,000–$10,000)

💵 TOP SPENDERS (ALL)
1. Bidder A — $20,000 (5 lots)
2. Bidder B — $15,000 (4 lots)
3. Bidder C — $10,000 (3 lots)

🆕 NEW BIDDERS — TOP SPENDERS
1. Bidder D — $8,000 (2 lots)
2. Bidder E — $6,000 (1 lot)
3. Bidder F — $4,000 (2 lots)

🔗 Full dashboard: [Power BI link]

In [ ]:
# %sql
# with hammer_price as
# (select hammer_price from
# company_db.gold.auction_sales_report_view
# where auction_id = 12345),

# number_of_lots as
# (select  number_of_lots from
# company_db.gold.auction_sales_report_view
# where auction_id = 12345),

# lots_sold as
# (select lots_sold from
#  company_db.gold.auction_sales_report_view
# where auction_id = 12345),

# number_of_units as
# (select sum(quantity) as number_of_units from company_db.gold.auction_results_fact 
# where auction_id = 12345),

# avg_price_per_unit as
# (select round(sum(hammerprice)/sum(quantity),0) as avg_price_per_unit from company_db.gold.auction_results_fact 
# where auction_id = 12345),

# bidders as
# (select registered_bidders, winning_bidders
# from 
# company_db.gold.auction_sales_report_view
# where auction_id = 12345),

# new_bidders AS
# (select count(distinct(user_id)) as new_bidders from 
# company_db.gold.bid_fact b
# where start_date = '2025-03-18' AND user_id IS NOT null
# AND
# NOT EXISTS
# (select 1 from 
# company_db.gold.bid_fact c
# where b.user_id = c.user_id AND
# start_date < '2025-03-18')),

# new_winners AS
# (select distinct(user_id) from 
# company_db.gold.bid_fact b
# where start_date = '2025-03-18' AND user_id IS NOT null
# AND
# NOT EXISTS
# (select 1 from 
# company_db.gold.bid_fact c
# where b.user_id = c.user_id AND
# start_date < '2025-03-18')),

# new_winners_count as
# (select count(*) as new_winners from new_winners
# where user_id IN
# (select user_id 
# from company_db.gold.auction_results_fact 
# where start_date = '2025-03-18')),

# no_show as 
# (select registered_bidders - online_bidders  AS no_show
# from 
# company_db.gold.auction_sales_report_view
# where start_date = '2025-03-18')

# select *
# from
# hammer_price,number_of_lots,lots_sold,number_of_units,avg_price_per_unit,bidders,new_bidders,new_winners_count,no_show;



In [ ]:
# %sql
# -- TOP 3 SOLD LOTS

# WITH top_3_sold_lots as
# (select 
# ld.item_name,
# af.lowestimate,
# af.highestimate,
# af.midestimate,
# af.hammerprice,
# af.favoritescount

# From
# company_db.gold.auction_results_fact as af
# INNER join
# company_db.gold.lots_dim ld
# ON af.lot_dim_key = ld.lot_dim_key
# where af.start_date = '2025-03-18' 
# order by hammerprice desc
# limit 3)

# select * from top_3_sold_lots

In [ ]:
# %sql
# -- TOP SPENDERS (ALL)
# WITH top_spenders AS
# (select 
# user_dim_key, sum(hammerprice) as total_spend,count(lot_id) as number_of_lots
# From 
# company_db.gold.auction_results_fact
# WHERE start_date = '2025-03-18'
# group by user_dim_key
# order by total_spend desc
# limit 5)

# SELECT 
# ud.full_name, top_spenders.total_spend, top_spenders.number_of_lots
# FROM top_spenders
# INNER JOIN company_db.gold.users_dim AS ud
# ON top_spenders.user_dim_key = ud.user_dim_key 


In [ ]:
# %sql
# -- NEW BIDDERS — TOP SPENDERS
# with new_bidders AS
# (select distinct(user_id) as user_id from 
# company_db.gold.bid_fact b
# where start_date = '2025-03-18' AND user_id IS NOT null
# AND
# NOT EXISTS
# (select 1 from 
# company_db.gold.bid_fact c
# where b.user_id = c.user_id AND
# start_date < '2025-03-18'))

# select 
# ud.full_name,
# sum(arf.hammerprice) as total_spend,
# count(lot_id) as number_of_lots
# From 
# new_bidders
# inner join company_db.gold.auction_results_fact as arf
# ON new_bidders.user_id = arf.user_id
# INNER JOIN company_db.gold.users_dim as ud
# ON new_bidders.user_id = ud.user_id
# where arf.start_date = '2025-03-18'
# group by ud.full_name
# order by sum(arf.hammerprice) desc
# limit 3;

In [ ]:
# %sql
# -- Estimates
# WITH binned_estimates AS
# (select 
# lot_dim_key,
# lot_number,
# lowestimate,
# midestimate,
# highestimate,
# hammerprice,
# CASE
# WHEN hammerprice > highestimate THEN 'ABOVE HIGH'
# WHEN hammerprice < lowestimate THEN 'BELOW LOW'
# ELSE 'IN RANGE' 
# END AS estimates
# from 
# company_db.gold.auction_results_fact
# where  start_date = '2024-03-19' and sold_flag =1),

# -- number of lots and hammered price below low estimates
# below_low AS
# (SELECT 
# count(lot_number) as lots_below_low, 
#         (SELECT 
#         count(lot_number) 
#         FROM binned_estimates) as number_of_lots,
#         round((lots_below_low/number_of_lots)*100,0) AS percentage_below
# FROM 
# binned_estimates
# where estimates = 'BELOW LOW'),


# above_high AS
# (SELECT 
# count(lot_number) as lots_above_high, 
#         (SELECT 
#         count(lot_number) 
#         FROM binned_estimates) as number_of_lots,
#         round((lots_above_high/number_of_lots)*100,0) AS percentage_above
# FROM 
# binned_estimates
# where estimates = 'ABOVE HIGH'),


# in_range AS
# (SELECT 
# count(lot_number) as in_range, 
#         (SELECT 
#         count(lot_number) 
#         FROM binned_estimates) as number_of_lots,
#         round((in_range/number_of_lots)*100,0) AS percentage_in_range
# FROM 
# binned_estimates
# where estimates = 'IN RANGE')


# SELECT 
# below_low.percentage_below,
# above_high.percentage_above,
# in_range.percentage_in_range
# FROM
# below_low, above_high, in_range